In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import os
import seaborn as sns
from torch.utils.tensorboard import SummaryWriter
import datetime
from sklearn.metrics import silhouette_score
import plotly.express as px
from sklearn.metrics import silhouette_samples, silhouette_score

In [ ]:
pip install gdown

In [ ]:
import gdown

# Google Drive file ID from the provided link
file_id = '1QQWSeku_wHSAZ5tKKceIGfVwGrLMR6hI'
output_path = 'OnlineRetail.csv'

gdown.download(f'https://drive.google.com/uc?id={file_id}', output_path, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1QQWSeku_wHSAZ5tKKceIGfVwGrLMR6hI
To: /content/OnlineRetail.csv
100%|██████████| 613k/613k [00:00<00:00, 62.9MB/s]


'OnlineRetail.csv'

In [ ]:
df = pd.read_csv('OnlineRetail.csv', encoding='latin1')
display(df.head())

,id_diagnostico,sector,tamano_empresa,porcentaje_procesos_documentados,presupuesto_anual_tecnología,respuesta_texto,nivel_madurez,recomendacion_principal
0,"DIAG_0000,Tecnología,Micro,0.05,3000000,""El tr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"DIAG_0001,Tecnología,Grande,0.81,261000000,""To...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DIAG_0002,Tecnología,Pequeña,0.27,47000000.0,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
3,DIAG_0003,Tecnología,Pequeña,0.30,19000000.0,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
4,"DIAG_0004,Manufactura,Micro,0.08,6000000,""Todo...",NaN,NaN,NaN,NaN,NaN,NaN,NaN


Se observa que algunas filas tienen múltiples valores combinados en la columna `id_diagnostico` y `sector` es `NaN` para estas filas. Esto indica que el parser de CSV no interpretó correctamente el delimitador para estas entradas. Procederemos a:

1.  Identificar las filas problemáticas donde `sector` es nulo.
2.  Dividir la columna `id_diagnostico` para estas filas en sus columnas correctas (ID, sector, tamaño_empresa, etc.).
3.  Actualizar el DataFrame con los valores corregidos.
4.  Convertir las columnas numéricas que puedan haber sido leídas como strings a tipos numéricos.

In [ ]:
# Identificar las filas donde 'sector' es NaN y 'id_diagnostico' contiene una coma, lo que indica un formato incorrecto.
malformed_mask = df['sector'].isna() & df['id_diagnostico'].astype(str).str.contains(',')

# Para estas filas, dividir la columna 'id_diagnostico' en varias columnas.
# n=5 asegura que se divida en 6 partes, manejando posibles comas dentro del campo 'respuesta_texto'.
split_data = df.loc[malformed_mask, 'id_diagnostico'].str.split(',', n=5, expand=True)

# Asignar nombres a las nuevas columnas temporales.
split_data.columns = [
    'temp_id_diagnostico',
    'temp_sector',
    'temp_tamano_empresa',
    'temp_porcentaje_procesos_documentados',
    'temp_presupuesto_anual_tecnologia',
    'temp_respuesta_texto'
]

# Actualizar el DataFrame original con los valores corregidos para las filas malformadas.
df.loc[malformed_mask, 'id_diagnostico'] = split_data['temp_id_diagnostico']
df.loc[malformed_mask, 'sector'] = split_data['temp_sector']
df.loc[malformed_mask, 'tamano_empresa'] = split_data['temp_tamano_empresa']
df.loc[malformed_mask, 'porcentaje_procesos_documentados'] = split_data['temp_porcentaje_procesos_documentados']
df.loc[malformed_mask, 'presupuesto_anual_tecnología'] = split_data['temp_presupuesto_anual_tecnologia']
df.loc[malformed_mask, 'respuesta_texto'] = split_data['temp_respuesta_texto']

# Convertir las columnas numéricas a su tipo de dato correcto. 'errors='coerce'' convertirá valores no numéricos a NaN.
df['porcentaje_procesos_documentados'] = pd.to_numeric(df['porcentaje_procesos_documentados'], errors='coerce')
df['presupuesto_anual_tecnología'] = pd.to_numeric(df['presupuesto_anual_tecnología'], errors='coerce')

display(df.head())

/tmp/ipykernel_1553/3579404469.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['0.05' '0.81' '0.08' '0.14' '0.46' '0.25' '0.05' '0.84' '0.15' '0.03'
 '0.15' '0.17' '0.05' '0.04' '0.01' '0.14' '0.81' '0.08' '0.29' '0.17'
 '0.35' '0.19' '0.42' '0.36' '0.17' '0.16' '0.16' '0.12' '0.08' '0.06'
 '0.16' '0.89' '0.28' '0.4' '0.07' '0.18' '0.1' '0.02' '0.42' '0.06'
 '0.01' '0.05' '0.08' '0.15' '0.17' '0.92' '0.41' '0.12' '0.34' '0.05'
 '0.12' '0.1' '0.07' '0.15' '0.04' '0.12' '0.11' '0.96' '0.12' '0.4'
 '0.13' '0.88' '0.02' '0.99' '0.04' '0.21' '0.93' '0.1' '0.16' '0.01'
 '0.1' '0.11' '0.06' '0.49' '0.29' '0.4' '0.9' '0.19' '0.1' '0.38' '0.2'
 '0.04' '0.3' '0.21' '0.1' '0.91' '0.13' '0.13' '0.2' '0.2' '0.08' '0.07'
 '0.4' '0.3' '0.14' '0.2' '0.16' '0.01' '0.18' '0.29' '0.34' '0.32' '0.05'
 '0.1' '0.1' '0.08' '0.02' '0.05' '0.18' '0.19' '0.07' '0.09' '0.86'
 '0.05' '0.96' '0.13' '0.9' '0.07' '0.48' '0.07' '

,id_diagnostico,sector,tamano_empresa,porcentaje_procesos_documentados,presupuesto_anual_tecnología,respuesta_texto,nivel_madurez,recomendacion_principal
0,DIAG_0000,Tecnología,Micro,0.05,3000000.0,"""El trabajo es muy empírico, no hay documentac...",NaN,NaN
1,DIAG_0001,Tecnología,Grande,0.81,261000000.0,"""Todo está automatizado, usamos datos para mej...",NaN,NaN
2,DIAG_0002,Tecnología,Pequeña,0.27,47000000.0,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
3,DIAG_0003,Tecnología,Pequeña,0.30,19000000.0,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
4,DIAG_0004,Manufactura,Micro,0.08,6000000.0,"""Todo es manual, dependemos de una sola person...",NaN,NaN


Ahora, confirmemos el estado de las columnas con `df.info()` para verificar los tipos de datos y los valores no nulos después de la limpieza.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 8 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id_diagnostico                    3000 non-null   object 
 1   sector                            3000 non-null   object 
 2   tamano_empresa                    3000 non-null   object 
 3   porcentaje_procesos_documentados  3000 non-null   float64
 4   presupuesto_anual_tecnología      3000 non-null   float64
 5   respuesta_texto                   3000 non-null   object 
 6   nivel_madurez                     2080 non-null   object 
 7   recomendacion_principal           2080 non-null   object 
dtypes: float64(2), object(6)
memory usage: 187.6+ KB


In [ ]:
import numpy as np
import pandas as pd
import io

# Identify rows where 'sector' is NaN and 'id_diagnostico' contains a comma, indicating a malformed format.
malformed_mask = df['sector'].isna() & df['id_diagnostico'].astype(str).str.contains(',')

# Create a list to store parsed rows from the malformed strings
parsed_data = []
parsed_indices = df.loc[malformed_mask].index

# Define expected column names for parsing individual CSV-like strings
col_names = [
    'p_id_diagnostico', 'p_sector', 'p_tamano_empresa',
    'p_porcentaje_procesos_documentados', 'p_presupuesto_anual_tecnologia',
    'p_respuesta_texto', 'p_nivel_madurez', 'p_recomendacion_principal'
]

# Process each malformed row using a CSV parser
for index in parsed_indices:
    malformed_string = df.loc[index, 'id_diagnostico']
    try:
        # Use io.StringIO to treat the string as a file and parse with pandas.read_csv
        # engine='python' is used for better handling of quoted fields with embedded commas.
        temp_df = pd.read_csv(io.StringIO(malformed_string), header=None, names=col_names, engine='python')
        parsed_data.append(temp_df.iloc[0]) # Append the first (and only) row
    except Exception as e:
        print(f"Error parsing string at index {index}: {malformed_string} - {e}")
        # If parsing fails, append NaNs to avoid breaking the process
        parsed_data.append(pd.Series([np.nan] * len(col_names), index=col_names))

if parsed_data:
    # Convert list of Series to DataFrame, preserving original indices
    split_data = pd.DataFrame(parsed_data, index=parsed_indices)

    # Convert numeric columns to float *before* assigning to the main DataFrame
    split_data['p_porcentaje_procesos_documentados'] = pd.to_numeric(split_data['p_porcentaje_procesos_documentados'], errors='coerce')
    split_data['p_presupuesto_anual_tecnologia'] = pd.to_numeric(split_data['p_presupuesto_anual_tecnologia'], errors='coerce')

    # Update the original DataFrame with the corrected values for the malformed rows.
    # Check if column exists in split_data before assigning, though it should if parsing was successful.
    df.loc[parsed_indices, 'id_diagnostico'] = split_data['p_id_diagnostico'].str.strip() if 'p_id_diagnostico' in split_data.columns else np.nan
    df.loc[parsed_indices, 'sector'] = split_data['p_sector'].str.strip() if 'p_sector' in split_data.columns else np.nan
    df.loc[parsed_indices, 'tamano_empresa'] = split_data['p_tamano_empresa'].str.strip() if 'p_tamano_empresa' in split_data.columns else np.nan
    df.loc[parsed_indices, 'porcentaje_procesos_documentados'] = split_data['p_porcentaje_procesos_documentados']
    df.loc[parsed_indices, 'presupuesto_anual_tecnología'] = split_data['p_presupuesto_anual_tecnologia']
    df.loc[parsed_indices, 'nivel_madurez'] = split_data['p_nivel_madurez'].str.strip() if 'p_nivel_madurez' in split_data.columns else np.nan
    df.loc[parsed_indices, 'recomendacion_principal'] = split_data['p_recomendacion_principal'].str.strip() if 'p_recomendacion_principal' in split_data.columns else np.nan

    # Special handling for 'respuesta_texto':
    df.loc[parsed_indices, 'respuesta_texto'] = split_data['p_respuesta_texto'].str.strip() if 'p_respuesta_texto' in split_data.columns else np.nan

    # Consistency check for malformed rows: If respuesta_texto is NaN for these, then also set
    # nivel_madurez and recomendacion_principal to NaN.
    mask_respuesta_texto_is_nan_in_malformed = df.loc[parsed_indices, 'respuesta_texto'].isna()
    df.loc[parsed_indices[mask_respuesta_texto_is_nan_in_malformed], ['nivel_madurez', 'recomendacion_principal']] = np.nan


# Convert numeric columns to their correct data type. 'errors='coerce'' will convert non-numeric values to NaN.
# This applies to all rows, not just the malformed ones, ensuring consistency.
df['porcentaje_procesos_documentados'] = pd.to_numeric(df['porcentaje_procesos_documentados'], errors='coerce')
df['presupuesto_anual_tecnología'] = pd.to_numeric(df['presupuesto_anual_tecnología'], errors='coerce')


# NEW LOGIC: Fix 'respuesta_texto' split into 'respuesta_texto' and 'nivel_madurez'
# and 'nivel_madurez' and 'recomendacion_principal' concatenated in 'recomendacion_principal'.
# This applies to rows that were not caught by the initial 'malformed_mask' if their
# 'sector' was not NaN, but subsequent fields are still misaligned.

# Identify rows that need correction for this specific misalignment pattern
# Conditions:
# 1. 'nivel_madurez' is not NaN AND contains more than 3 words (indicating it's part of 'respuesta_texto')
# 2. 'recomendacion_principal' is not NaN AND contains a comma (indicating it's concatenated nivel_madurez + recomendacion_principal)
mask_needs_re_split_fields = (
    df['nivel_madurez'].notna() &
    (df['nivel_madurez'].astype(str).str.strip().str.count(' ') > 2) & # More than 3 words implies part of respuesta_texto
    df['recomendacion_principal'].notna() &
    df['recomendacion_principal'].astype(str).str.contains(',') # Contains comma, implies concatenated fields
)

if mask_needs_re_split_fields.any():
    # Store original values for problematic rows to ensure correct splitting
    original_respuesta_texto_problematic = df.loc[mask_needs_re_split_fields, 'respuesta_texto'].astype(str)
    original_nivel_madurez_problematic = df.loc[mask_needs_re_split_fields, 'nivel_madurez'].astype(str)
    original_recomendacion_principal_problematic = df.loc[mask_needs_re_split_fields, 'recomendacion_principal'].astype(str)

    # 1. Reconstruct 'respuesta_texto' by combining the two parts
    df.loc[mask_needs_re_split_fields, 'respuesta_texto'] = (
        original_respuesta_texto_problematic + ', ' + original_nivel_madurez_problematic
    ).str.strip()

    # 2. Split 'recomendacion_principal' into its correct 'nivel_madurez' and 'recomendacion_principal'
    #    using the first comma as a delimiter.
    split_recommendation_parts = original_recomendacion_principal_problematic.str.split(',', n=1, expand=True)

    # Assign the first part (correct nivel_madurez)
    df.loc[mask_needs_re_split_fields, 'nivel_madurez'] = split_recommendation_parts[0].str.strip()

    # Assign the second part (correct recomendacion_principal)
    if 1 in split_recommendation_parts.columns:
        df.loc[mask_needs_re_split_fields, 'recomendacion_principal'] = split_recommendation_parts[1].str.strip()
    else:
        # If there's no second part (only nivel_madurez was present), set recomendacion_principal to NaN
        df.loc[mask_needs_re_split_fields, 'recomendacion_principal'] = np.nan


# Final consistency check across the entire DataFrame:
# If 'respuesta_texto' is NaN, ensure 'nivel_madurez' and 'recomendacion_principal' are also NaN.
mask_respuesta_texto_is_nan_final = df['respuesta_texto'].isna()
df.loc[mask_respuesta_texto_is_nan_final, ['nivel_madurez', 'recomendacion_principal']] = np.nan

# Remove all quotes from all string columns (this was the last step in the previous turn)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.replace('"', '', regex=False).str.replace('\'', '', regex=False).replace('nan', np.nan)

display(df.head())

,id_diagnostico,sector,tamano_empresa,porcentaje_procesos_documentados,presupuesto_anual_tecnología,respuesta_texto,nivel_madurez,recomendacion_principal
0,DIAG_0000,Tecnología,Micro,0.05,3000000.0,"El trabajo es muy empírico, no hay documentaci...",Inicial,Mapear flujos de trabajo de desarrollo y usar ...
1,DIAG_0001,Tecnología,Grande,0.81,261000000.0,"Todo está automatizado, usamos datos para mejo...",Optimizado,Aplicar MLOps y analítica predictiva para opti...
2,DIAG_0002,Tecnología,Pequeña,0.27,47000000.0,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
3,DIAG_0003,Tecnología,Pequeña,0.30,19000000.0,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
4,DIAG_0004,Manufactura,Micro,0.08,6000000.0,"Todo es manual, dependemos de una sola persona...",Inicial,Estandarizar el registro diario de mermas e in...


In [41]:
df.to_csv('cleaned_data.csv', index=False, encoding='utf-8')
print('DataFrame exportado a cleaned_data.csv')

DataFrame exportado a cleaned_data.csv


In [42]:
from google.colab import files

files.download('cleaned_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>